# Импорты

In [1]:
import pandas as pd
import numpy as np
import scipy

# Инициализация задачи

In [2]:
n = 50

df = pd.DataFrame({
    "x_i": [65, 68, 71, 74, 77, 80, 83, 86, 89],
    "x_i+1": [68, 71, 74, 77, 80, 83, 86, 89, 92],
    "m_i": [3, 5, 6, 7, 9, 7, 6, 5, 2]
})
df

,x_i,x_i+1,m_i
0,65,68,3
1,68,71,5
2,71,74,6
3,74,77,7
4,77,80,9
5,80,83,7
6,83,86,6
7,86,89,5
8,89,92,2


# Середина интервала

In [3]:
df["average"] = (df["x_i"] + df["x_i+1"]) / 2
df

,x_i,x_i+1,m_i,average
0,65,68,3,66.5
1,68,71,5,69.5
2,71,74,6,72.5
3,74,77,7,75.5
4,77,80,9,78.5
5,80,83,7,81.5
6,83,86,6,84.5
7,86,89,5,87.5
8,89,92,2,90.5


# Выборочные средняя, дисперсия, среднеквадратичное отклонение и исправленная выборочная дисперсия

In [4]:
_x = sum(df["average"] * df["m_i"]) / n
D_x = sum(df["average"] ** 2 * df["m_i"]) / n - _x ** 2
sigma_x = np.sqrt(D_x)
S_2 = sum((df["average"] - _x) ** 2 * df["m_i"]) / (n - 1)

# $\displaystyle x_i - \={x} \\ x_{i + 1} - \={x} \\ \frac{x_i - \={x}}{\sigma} \\ \frac{x_{i + 1} - \={x}}{\sigma} \\ \Phi \left( \frac{x_i - \={x}}{\sigma} \right) \\ \Phi \left( \frac{x_{i + 1} - \={x}}{\sigma} \right) \\ P_i \\ m'_i \\ m_i - m'_i \\ (m_i - m'_i)^2 \\ \frac{(m_i - m'_i)^2}{m_i} \\ m_i^2 \\ \frac{m_i^2}{m'_i} $

In [5]:
df["x_i - _x"] = df["x_i"] - _x
df.loc[df.index[0], "x_i - _x"] = -float("inf")

df["x_i+1 - _x"] = df["x_i+1"] - _x
df.loc[df.index[-1], "x_i+1 - _x"] = float("inf")

df["x_i - _x / sigma"] = df["x_i - _x"] / sigma_x
df["x_i+1 - _x / sigma"] = df["x_i+1 - _x"] / sigma_x

df["Ф_i"] = scipy.stats.norm.cdf(df["x_i - _x / sigma"]) - 0.5
df["Ф_i+1"] = scipy.stats.norm.cdf(df["x_i+1 - _x / sigma"]) - 0.5

df["P_i"] = df["Ф_i+1"] - df["Ф_i"]
df["m'_i"] = n * df["P_i"]

df["m_i - m'_i"] = df["m_i"] - df["m'_i"]
df["(m_i - m'_i)^2"] = df["m_i - m'_i"] ** 2
df["(m_i - m'_i)^2 / m_i"] = df["(m_i - m'_i)^2"] / df["m_i"]

df["m_i^2"] = df["m_i"] ** 2
df["m_i^2 / m'_i"] = df["m_i^2"] / df["m'_i"]

df

,x_i,x_i+1,m_i,average,x_i - _x,x_i+1 - _x,x_i - _x / sigma,x_i+1 - _x / sigma,Ф_i,Ф_i+1,P_i,m'_i,m_i - m'_i,(m_i - m'_i)^2,(m_i - m'_i)^2 / m_i,m_i^2,m_i^2 / m'_i
0,65,68,3,66.5,-inf,-10.26,-inf,-1.588791,-0.500000,-0.443946,0.056054,2.802688,0.197312,0.038932,0.012977,9,3.211203
1,68,71,5,69.5,-10.26,-7.26,-1.588791,-1.124232,-0.443946,-0.369543,0.074403,3.720173,1.279827,1.637957,0.327591,25,6.720118
2,71,74,6,72.5,-7.26,-4.26,-1.124232,-0.659674,-0.369543,-0.245268,0.124274,6.213722,-0.213722,0.045677,0.007613,36,5.793629
3,74,77,7,75.5,-4.26,-1.26,-0.659674,-0.195115,-0.245268,-0.077348,0.167920,8.395995,-1.395995,1.948802,0.278400,49,5.836116
4,77,80,9,78.5,-1.26,1.74,-0.195115,0.269444,-0.077348,0.106206,0.183554,9.177723,-0.177723,0.031585,0.003509,81,8.825718
5,80,83,7,81.5,1.74,4.74,0.269444,0.734003,0.106206,0.268527,0.162321,8.116025,-1.116025,1.245512,0.177930,49,6.037438
6,83,86,6,84.5,4.74,7.74,0.734003,1.198562,0.268527,0.384651,0.116124,5.806214,0.193786,0.037553,0.006259,36,6.200254
7,86,89,5,87.5,7.74,10.74,1.198562,1.663121,0.384651,0.451856,0.067205,3.360252,1.639748,2.688773,0.537755,25,7.439918
8,89,92,2,90.5,10.74,inf,1.663121,inf,0.451856,0.500000,0.048144,2.407207,-0.407207,0.165818,0.082909,4,1.661677


# Итого

In [7]:
print(
    " m_i: ", sum(df["m_i"]), "\n",
    "P_i: ", sum(df["P_i"]), "\n",
    "(m_i - m'_i)^2 / m_i: ", sum(df["(m_i - m'_i)^2 / m_i"]), "\n",
    "m_i^2 / m'_i: ", sum(df["m_i^2 / m'_i"]), "\n"
)

 m_i:  50 
 P_i:  1.0 
 (m_i - m'_i)^2 / m_i:  1.4349439977277818 
 m_i^2 / m'_i:  51.726069881232384 

